# 03_main_run.ipynb — 結果出力パイプライン本処理

**役割**: チャット1の `records.json` とチャット2の `classifications.json` を結合し、派生指標を計算して 3 形式の CSV を出力する。

## 入力
- `outputs/prepared_data/{TIMESTAMP}_records.json` (チャット1の出力)
- `outputs/dify_results/{TIMESTAMP}_classifications.json` (チャット2の出力)

## 出力
- `outputs/final/{TIMESTAMP}_wide.csv` — Tableau ダッシュボード用
- `outputs/final/{TIMESTAMP}_long.csv` — 環境要因縦展開、ドリルダウン用
- `outputs/final/{TIMESTAMP}_aggregated.csv` — DB 追記用 (repair_id 単位)
- `outputs/final/{TIMESTAMP}_output_meta.json` — 出力メタ情報

## ステップ
1. 設定 / パス確定
2. 入力 JSON 読み込み
3. 結合 (`merge_records_and_classifications`)
4. 派生指標計算 (`calculate_derived_metrics`)
5. 環境フラグ展開 + コード名付与
6. 3 形式 CSV 出力 (`write_all_formats`)
7. 実行サマリ表示

## 1. 設定 / パス確定

`TIMESTAMP` を必要に応じて書き換える。最新ファイルを自動取得するロジックも用意してある。

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

# プロジェクトルートとモジュールパス
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

YAML_PATH = REPO_ROOT / "config" / "classification_codes.yaml"
PREPARED_DIR = REPO_ROOT / "outputs" / "prepared_data"
DIFY_DIR = REPO_ROOT / "outputs" / "dify_results"
FINAL_DIR = REPO_ROOT / "outputs" / "final"

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"YAML_PATH exists: {YAML_PATH.exists()}")
print(f"PREPARED_DIR exists: {PREPARED_DIR.exists()}")
print(f"DIFY_DIR exists: {DIFY_DIR.exists()}")

In [ ]:
# TIMESTAMP の決定:
#   明示指定したい場合は TIMESTAMP = "20260513_120000" のように書き換える
#   None のままなら prepared_data 配下の最新 records.json から自動取得
TIMESTAMP: str | None = None

if TIMESTAMP is None:
    candidates = sorted(PREPARED_DIR.glob("*_records.json"))
    if not candidates:
        raise FileNotFoundError(
            f"records.json が {PREPARED_DIR} に見つかりません。"
            "チャット1を実行するか、TIMESTAMP を明示してください。"
        )
    latest = candidates[-1]
    TIMESTAMP = latest.name.removesuffix("_records.json")
    print(f"TIMESTAMP 自動検出: {TIMESTAMP}")

RECORDS_PATH = PREPARED_DIR / f"{TIMESTAMP}_records.json"
CLASSIFICATIONS_PATH = DIFY_DIR / f"{TIMESTAMP}_classifications.json"

print(f"records:         {RECORDS_PATH} ({'OK' if RECORDS_PATH.exists() else 'MISSING'})")
print(f"classifications: {CLASSIFICATIONS_PATH} ({'OK' if CLASSIFICATIONS_PATH.exists() else 'MISSING'})")

## 2. 入力 JSON 読み込み

In [ ]:
from codes_loader import load_codes

codebook = load_codes(YAML_PATH)
print(f"codebook version: {codebook.meta.version}")
print(f"ML codes:   {len(codebook.get_failure_codes_for_product('ML'))}")
print(f"LENS codes: {len(codebook.get_failure_codes_for_product('LENS'))}")

In [ ]:
records = json.loads(RECORDS_PATH.read_text(encoding="utf-8"))
classifications = json.loads(CLASSIFICATIONS_PATH.read_text(encoding="utf-8"))

print(f"records: {len(records)} 件")
print(f"classifications: {len(classifications)} 件")

if len(records) != len(classifications):
    print(f"⚠ 件数差: {abs(len(records) - len(classifications))} 件 (Difyで失敗したバッチの可能性)")

# スキーマ確認 (最初の1件)
print("\n--- records[0] keys ---")
print(list(records[0].keys()))
print("\n--- classifications[0] keys ---")
print(list(classifications[0].keys()))

## 3-5. 結合 → 派生指標 → 環境フラグ展開 → コード名付与

`build_derived_dataframe` でまとめて実行する。

In [ ]:
from derive_metrics import build_derived_dataframe

df = build_derived_dataframe(records, classifications, codebook)

print(f"結合後行数: {len(df)}")
print(f"カラム数: {len(df.columns)}")
print(f"unique repair_id: {df['repair_id'].nunique()}")
df.head(3)

### 派生指標カラムの一覧確認

In [ ]:
derived_cols = [
    "perspective_match", "is_misjudged",
    "is_manufacturer_responsibility_user",
    "is_manufacturer_responsibility_repair",
    "has_harsh_env", "has_repair_confirmed_env",
    "is_service_record", "min_confidence",
]
missing = [c for c in derived_cols if c not in df.columns]
assert not missing, f"派生指標カラム欠落: {missing}"
df[derived_cols].head(5)

## 6. 3 形式 CSV 出力

In [ ]:
from output_formatter import write_all_formats

paths = write_all_formats(
    df,
    output_dir=FINAL_DIR,
    timestamp=TIMESTAMP,
    codebook=codebook,
)

for kind, p in paths.items():
    size_kb = p.stat().st_size / 1024
    print(f"{kind:12s}: {p}  ({size_kb:.1f} KB)")

## 7. 実行サマリ

In [ ]:
meta = json.loads(paths["meta"].read_text(encoding="utf-8"))
print(json.dumps(meta, ensure_ascii=False, indent=2))

### 派生指標の出現率サマリ (テーブル表示)

In [ ]:
summary_rows = []
for col in derived_cols:
    if col == "min_confidence":
        continue
    s = df[col].fillna(False).astype(bool)
    summary_rows.append({
        "metric": col,
        "true_count": int(s.sum()),
        "total": int(len(s)),
        "rate": round(s.mean(), 4),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
# 製品種別ごとの故障コード Top 10 (修理者視点)
for pt in df["product_type"].dropna().unique():
    print(f"\n--- {pt} 修理者視点コード Top 10 ---")
    top = (
        df[df["product_type"] == pt]
        .groupby(["repair_failure_code", "repair_failure_name"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(10)
    )
    print(top.to_string(index=False))

In [ ]:
# 環境要因の出現頻度
env_flag_cols = [c for c in df.columns if c.startswith("has_") and c not in ("has_harsh_env", "has_repair_confirmed_env")]
env_counts = df[env_flag_cols].astype(bool).sum().sort_values(ascending=False)
print("--- 環境要因の出現件数 ---")
print(env_counts.to_string())

## 完了

出力 CSV を Tableau Desktop で開いて可視化作業へ進む。
詳細な品質レビューは `04_quality_review.ipynb` で行う。